# KRAS-G12D / A\*11:01 binder retry — **ODesign** (the different-generator lever)

**Why this run.** PXDesign bounded the KRAS-G12D external binder at **0/80 dual-oracle passers** because AF2-IG
and Protenix **anti-correlate** (r=−0.41) — no design satisfies both. ODesign is an **all-atom interaction
world-model, NOT AF2-based** → a genuinely different generative prior, with **explicit epitope/hotspot control**
(force the binder onto the G12D Asp). The one principled shot left at the external key.
*(Repo: The-Institute-for-AI-Molecular-Design/ODesign, Apache-2.0, arXiv 2510.22304.)*

**Fully automatic — no uploads, no Google-Drive file IDs:**
- Target pMHC PDB + input JSON: **`wget` from the public repo**.
- **CCD file** (528MB): **`wget` from this repo's GitHub Release** (`odesign-ccd-v20240608`).
- ODesign **checkpoints**: from HuggingFace.
- All heavy assets are **cached in YOUR Drive** (`MyDrive/cancer-recon/odesign_assets/`) on the first run, so
  every later run reads them from Drive — no re-download.
- ⇒ Set a GPU and **Run-all**. That's it.

> **Honest banner (rule 7).** Heavy CUDA-12.1 stack (torch 2.3.1 + pyg + deepspeed). I could **not** dry-run
> the CUDA wheels on the M2 (platform-specific), so treat the first run as possibly needing install iteration.
> If pip fights you, the maintainers' **container path** (`RUN_GUIDE.md` Path A) is the lower-risk option.

**Runtime:** set **Runtime → Change runtime type → T4 GPU** BEFORE running — the first cell refuses to proceed
without a GPU (a CPU fallback runs ~50× slower and silently wastes hours).

In [ ]:
# --- GPU GUARD (before install; torch not present yet, so probe nvidia-smi) ---
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("NO GPU detected. Runtime -> Change runtime type -> T4 GPU, then re-run this cell.")
print(r.stdout[:600])
print("GPU present — proceeding to install.")

### If Colab prompts to RESTART after the install
The full `requirements.txt` pins `numpy==1.26.3` / `protobuf==3.20.2` etc., which can downgrade Colab's base and
trigger a restart prompt. **That's fine** — click restart, then **re-run from the next cell down** (skip the
install cell). Do NOT re-run install. (Rule 7: never `condacolab` here — it would wipe the pip installs.)

In [ ]:
%cd /content
!rm -rf /content/ODesign
!git clone --depth 1 https://github.com/The-Institute-for-AI-Molecular-Design/ODesign.git
%cd /content/ODesign
!pip install -q -r requirements.txt -f https://data.pyg.org/whl/torch-2.3.1+cu121.html

In [ ]:
# --- GPU GUARD (after install; fail LOUD before spending time) ---
import torch
assert torch.cuda.is_available(), "NO GPU after install — set runtime to T4 and re-run from here."
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU", torch.cuda.get_device_name(0))

In [ ]:
# --- Mount YOUR Drive for a persistent asset cache (checkpoints + CCD download ONCE) ---
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ASSETS = "/content/drive/MyDrive/cancer-recon/odesign_assets"
    print("Drive mounted — asset cache:", ASSETS)
except Exception as e:
    ASSETS = "/content/odesign_assets"
    print("no Drive (", type(e).__name__, ") — assets EPHEMERAL (re-download each session):", ASSETS)
for d in (ASSETS + "/ckpt", ASSETS + "/data", "/content/ODesign/ckpt", "/content/ODesign/data"):
    os.makedirs(d, exist_ok=True)

In [ ]:
# --- Checkpoints + CCD: read from Drive cache if present; else fetch ONCE and cache to Drive ---
import os, glob, shutil, subprocess
RELEASE = "https://github.com/AnshumanAtrey/cancer-recog-apoptosis/releases/download/odesign-ccd-v20240608"

# 1) checkpoints (design model + inverse-folding + ProteinMPNN) — Drive cache or HuggingFace once
KEY = "odesign_base_prot_flex.pt"
if os.path.exists(ASSETS + "/ckpt/" + KEY):
    for f in glob.glob(ASSETS + "/ckpt/*"):
        shutil.copy(f, "/content/ODesign/ckpt/")
    print("checkpoints: loaded from Drive cache (no re-download) ✓")
else:
    print("checkpoints: downloading once (HuggingFace) ...")
    subprocess.run(["bash", "./ckpt/get_odesign_ckpt.sh", "./ckpt"], cwd="/content/ODesign")
    for f in glob.glob("/content/ODesign/ckpt/*"):
        shutil.copy(f, ASSETS + "/ckpt/")
    print("checkpoints: downloaded + cached to Drive ✓")

# 2) CCD files — Drive cache, else wget from THIS repo's GitHub Release (automatic, no file IDs)
CIF = "/content/ODesign/data/components.v20240608.cif"
PKL = "/content/ODesign/data/components.v20240608.cif.rdkit_mol.pkl"
cif_c, pkl_c = ASSETS + "/data/" + os.path.basename(CIF), ASSETS + "/data/" + os.path.basename(PKL)
if os.path.exists(cif_c) and os.path.exists(pkl_c):
    shutil.copy(cif_c, CIF); shutil.copy(pkl_c, PKL)
    print("CCD: loaded from Drive cache ✓")
else:
    print("CCD: downloading from GitHub Release (one-time, ~528MB) ...")
    subprocess.run(["wget", "-q", "-O", CIF, RELEASE + "/components.v20240608.cif"])
    subprocess.run(["wget", "-q", "-O", PKL, RELEASE + "/components.v20240608.cif.rdkit_mol.pkl"])
    assert os.path.getsize(CIF) > 1_000_000 and os.path.getsize(PKL) > 1_000_000, "CCD download failed"
    shutil.copy(CIF, cif_c); shutil.copy(PKL, pkl_c)   # cache to Drive for next time
    print("CCD: downloaded from release + cached to Drive ✓")
print("ckpt dir:", os.listdir("/content/ODesign/ckpt"))

In [ ]:
# Pull the validated INPUT directly from the public repo (no manual upload — the files are tracked).
import os
os.makedirs("/content/ODesign/examples/protein_design/prot_binding_prot", exist_ok=True)
RAW = "https://raw.githubusercontent.com/AnshumanAtrey/cancer-recog-apoptosis/main"
print("fetching inputs from", RAW)

In [ ]:
!wget -q -O /content/ODesign/data/kras_g12d_A1101_free_mut_cropped.pdb {RAW}/runs/rung30_kras_g12d/staging/kras_g12d_A1101_free_mut_cropped.pdb
!wget -q -O /content/ODesign/examples/protein_design/prot_binding_prot/kras.json {RAW}/runs/rung30_kras_g12d/odesign/kras_odesign_input.json

In [ ]:
import os, json
spec = json.load(open("/content/ODesign/examples/protein_design/prot_binding_prot/kras.json"))
print("input JSON OK:", json.dumps(spec, indent=2))
p = "/content/ODesign/data/kras_g12d_A1101_free_mut_cropped.pdb"
assert os.path.exists(p) and os.path.getsize(p) > 1000, "target PDB fetch failed (check the RAW url / branch)"
print("target PDB OK:", os.path.getsize(p), "bytes")

In [ ]:
# --- INFERENCE with GPU guard + heartbeat (inference is one long subprocess; rule 7) ---
import torch, os, time, threading, subprocess
assert torch.cuda.is_available(), "NO GPU — abort (CPU would be ~50x slower and silent)."
os.chdir("/content/ODesign")
OUT = "/content/ODesign/outputs"
os.makedirs(OUT, exist_ok=True)

def heartbeat(stop):
    t0 = time.time()
    while not stop.is_set():
        n = sum(len(fs) for _, _, fs in os.walk(OUT))
        try:
            free, total = torch.cuda.mem_get_info(); used = (total - free) / 1e9
        except Exception:
            used = -1.0
        print(f"[hb] {time.time()-t0:6.0f}s  outputs={n} files  GPU_used={used:.1f}GB", flush=True)
        stop.wait(30)

cmd = ["python", "./scripts/inference.py",
       "exp=train_odesign_base_prot_flex",
       "data_root_dir=./data", "ckpt_root_dir=./ckpt",
       "exp.infer_model_name=odesign_base_prot_flex",
       "exp.design_modality=protein",
       "exp.input_json_path=./examples/protein_design/prot_binding_prot/kras.json",
       "exp.exp_name=kras_g12d_odesign_v1",
       "exp.seeds=[42,123,777,2024,31337]",
       "exp.model.sample_diffusion.N_sample=10",
       "exp.use_msa=false", "exp.num_workers=4",
       "exp.model.inference_noise_schedulers.coordinate.partial_diffusion.enable=false",
       "exp.model.inference_noise_schedulers.coordinate.partial_diffusion.snr=0.1"]
print("RUN:", " ".join(cmd), flush=True)
stop = threading.Event()
th = threading.Thread(target=heartbeat, args=(stop,), daemon=True); th.start()
proc = subprocess.run(cmd)
stop.set(); time.sleep(0.1)
print("inference.py exit code:", proc.returncode,
      "| outputs:", sum(len(fs) for _, _, fs in os.walk(OUT)), "files")
assert proc.returncode == 0, "inference failed — read the traceback above (often a CCD/ckpt path or a pyg/torch mismatch)."

In [ ]:
# Persist outputs to your Drive BEFORE the runtime dies (rule 7)
import shutil, os
dst = ASSETS.rsplit("/", 1)[0] + "/odesign_kras_g12d_v1"   # MyDrive/cancer-recon/odesign_kras_g12d_v1
os.makedirs(dst, exist_ok=True)
shutil.make_archive(dst + "/outputs", "zip", "/content/ODesign/outputs")
print("saved:", dst + "/outputs.zip")

## ✅ Next: the real test — MUT-vs-WT discrimination scoring (separate step)
ODesign wrote designed binders (`outputs/.../*.cif`) against the **MUT** pMHC. Binding isn't the win — it must
**discriminate**. For each top binder sequence:
1. Fold it against the **MUT** pMHC (`kras_g12d_A1101_free_mut_pmhc.pdb`) **and** the **WT** pMHC
   (`kras_g12d_A1101_free_wt_pmhc.pdb`) — on **both** Protenix and AF2-IG (the bound was dual-certification).
   Both PDBs are in the repo under `runs/rung30_kras_g12d/staging/` (the notebook can wget them too).
2. **Win = high on MUT, low on WT, on BOTH oracles.** A binder that grips MUT and WT equally fails (it would
   attack the WT peptide on normal tissue, R32).
3. Use the existing scoring path (`notebooks/binder_score_colab.ipynb` / `binder_specificity_*`).

**Either outcome is informative:** a dual-certified discriminating binder = a breakthrough artifact; another 0
= a strong second-generator confirmation the external-binder route is bounded in-silico → recognition rests on
the validated **internal key** (R27/R33/R34–40).